# Configuration

Every agent so far has run on the same defaults. Underneath, `invoke` is built on three replaceable pieces — the **LLM** it queries, the **REPL** its code runs in, and the **protocol** that translates between them.

This tutorial is about changing them, globally and per call.

## Setup

Install JAZ, provide an API key, and pick a model. Everything below runs in Colab — no local install.

In [1]:
%pip install -q jaz-lang

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, getpass

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')

In [3]:
import jaz
from jaz.llm import LiteLLM

jaz.configure(llm=LiteLLM(model='openai/gpt-5.4-nano'))

## What you are running on

`jaz.configure(...)` sets the global defaults — you have called it in the setup cell of every tutorial. `jaz.get_config()` reads them back.

In [4]:
import jaz

config = jaz.get_config()

print("llm     ", type(config.llm).__name__)
print("repl    ", type(config.repl).__name__)
print("protocol", type(config.protocol).__name__)

llm      LiteLLM
repl     PythonREPL
protocol CodeOnlyProtocol


## The REPL decides what the agent may do

The REPL is the sandbox the agent's code runs in, and it is where the limits on that code live: execution timeout, memory, readable and writable paths — and which modules may be imported.

**By default the agent may not import anything at all.** That is a default, not a law of the framework, and it is worth seeing because it is stated to the model in its system prompt:

In [5]:
from jaz import invoke
from jaz.hooks import Hook
from jaz.repl import PythonREPL


class ShowImportRule(Hook):
    """Prints the line of the system prompt that governs imports."""

    def __init__(self):
        self.done = False

    def on_llm_query_enter(self, event):
        if not self.done:
            for message in event.messages:
                if message["role"] == "system":
                    for line in message["content"].splitlines():
                        if "import" in line.lower():
                            print("   ", line.strip())
                    self.done = True
                    break
        return []


print("default REPL:")
jaz.configure(repl=PythonREPL())
with ShowImportRule():
    invoke(task="Return the number 1.")

default REPL:


    - You are NOT allowed to import any Python modules.


Now allow a couple of modules. The rule the model is given changes with the configuration — it is not a fixed instruction:

In [6]:
print("repl=PythonREPL(allowed_imports=['math', 'statistics']):")
jaz.configure(repl=PythonREPL(allowed_imports=["math", "statistics"]))
with ShowImportRule():
    result = invoke(task="Import math and return math.floor(7.8) as an int.")

print("   result:", result)

repl=PythonREPL(allowed_imports=['math', 'statistics']):
    - You may ONLY import Python modules whose name matches:


   result: 7


So “the agent cannot import” is really “this REPL was configured not to let it”. Widen `allowed_imports` when the agent genuinely needs a library; keep it narrow when you would rather hand the library in as an input, as tutorial 1 did.

## Changing configuration for one call

`ConfigOverride` is to configuration what a hook is to behaviour, and it is used the same two ways: **positionally** for a single `invoke`, or as a **context manager** for everything inside it, nested invokes included.

In [7]:
from jaz import ConfigOverride

# Back to a locked-down default for everything else.
jaz.configure(repl=PythonREPL())

# ConfigOverride works the two ways a hook does. Positionally it applies to one call:
loose = ConfigOverride(repl=PythonREPL(allowed_imports=["math"]))

with ShowImportRule():
    invoke(loose, task="Return the number 1.")

print("...and the global default is untouched:")
with ShowImportRule():
    invoke(task="Return the number 1.")

    - You may ONLY import Python modules whose name matches:


...and the global default is untouched:
    - You are NOT allowed to import any Python modules.


The override applied to the first call and left the global default alone.

## The rest of the REPL's settings

In [8]:
# The REPL carries the other execution limits too.
strict = PythonREPL(exec_timeout=5.0)

print("exec_timeout:", strict.exec_timeout, "seconds")
print("allowed_imports:", strict.allowed_imports)

exec_timeout: 5.0 seconds
allowed_imports: ()


## What to take away

- **Three replaceable pieces**: `llm`, `repl`, `protocol`. `jaz.configure()` sets the defaults; `jaz.get_config()` reads them.
- **The REPL is the sandbox.** Timeouts, memory, file access and imports are its settings, not fixed rules.
- **Imports are off by default and configurable** via `PythonREPL(allowed_imports=[...])`. The model is told the current rule.
- **`ConfigOverride`** changes configuration for one call (positional) or a whole block (`with`), mirroring how hooks are scoped.

---

**Next:** [Putting it together](05-putting-it-together.ipynb) — one real problem, using all of it at once.